In [0]:
# Demo: Building Dashboards & Visualizations — Setup
# Creates FOUR types of data assets for dashboard datasets:
#   1. Star Schema (fact + dimension tables)
#   2. Pre-Joined Gold Table
#   3. SQL Views
#   4. Metric View

import re
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DoubleType
from datetime import date, timedelta
import random

# --- Schema setup ---
username = spark.sql("SELECT current_user()").collect()[0][0]
clean_username = re.sub(r'[^a-zA-Z0-9]', '_', username.split('@')[0])
catalog = "workspace"
schema = f"demo_dashboards_{clean_username}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`")
spark.sql(f"USE CATALOG `{catalog}`")
spark.sql(f"USE SCHEMA `{schema}`")

In [0]:
# =============================================================
# ASSET TYPE 1: STAR SCHEMA (Fact + Dimension Tables)
# =============================================================
# A star schema separates descriptive attributes (dimensions) from
# transactional measures (facts). This is the classic data warehouse
# pattern and provides maximum flexibility for ad-hoc analysis.

random.seed(42)

# --- Shared reference data ---
regions_data = [
    (1, 'Northeast', 'East'),
    (2, 'Southeast', 'East'),
    (3, 'Midwest', 'Central'),
    (4, 'West', 'West'),
    (5, 'Northwest', 'West')
]

products_data = [
    (1, 'Laptop Pro', 'Electronics', 'Computing'),
    (2, 'Tablet Air', 'Electronics', 'Computing'),
    (3, 'Wireless Earbuds', 'Electronics', 'Audio'),
    (4, 'Smart Watch', 'Electronics', 'Wearables'),
    (5, 'Denim Jacket', 'Clothing', 'Outerwear'),
    (6, 'Running Shoes', 'Clothing', 'Footwear'),
    (7, 'Winter Coat', 'Clothing', 'Outerwear'),
    (8, 'Standing Desk', 'Home & Garden', 'Furniture'),
    (9, 'LED Lamp', 'Home & Garden', 'Lighting'),
    (10, 'Yoga Mat', 'Sports', 'Fitness'),
    (11, 'Dumbbell Set', 'Sports', 'Fitness'),
    (12, 'Protein Bars', 'Food & Beverage', 'Nutrition'),
    (13, 'Coffee Beans', 'Food & Beverage', 'Beverages'),
    (14, 'Garden Hose', 'Home & Garden', 'Outdoor'),
    (15, 'Tennis Racket', 'Sports', 'Equipment')
]

customers_data = []
first_names = ['James', 'Mary', 'Robert', 'Patricia', 'John', 'Jennifer', 'Michael', 'Linda', 'David', 'Sarah']
last_names = ['Smith', 'Johnson', 'Williams', 'Brown', 'Jones', 'Garcia', 'Miller', 'Davis', 'Wilson', 'Taylor']
segments = ['Enterprise', 'Mid-Market', 'SMB', 'Consumer']
channels = ['Online', 'Retail Store', 'Partner', 'Mobile App']

for i in range(1, 201):
    fname = random.choice(first_names)
    lname = random.choice(last_names)
    signup = date(2020, 1, 1) + timedelta(days=random.randint(0, 1460))
    customers_data.append((i, f"{fname} {lname}", random.choice(segments), random.choice(regions_data)[0], signup))

# --- dim_region ---
spark.sql("DROP TABLE IF EXISTS dim_region")
df_region = spark.createDataFrame(regions_data, schema=["region_id", "region_name", "territory"])
df_region.write.saveAsTable("dim_region")
print(f"✓ dim_region: {df_region.count()} rows")

# --- dim_product ---
spark.sql("DROP TABLE IF EXISTS dim_product")
df_product = spark.createDataFrame(products_data, schema=["product_id", "product_name", "category", "subcategory"])
df_product.write.saveAsTable("dim_product")
print(f"✓ dim_product: {df_product.count()} rows")

# --- dim_customer ---
spark.sql("DROP TABLE IF EXISTS dim_customer")
df_customer = spark.createDataFrame(customers_data, schema=["customer_id", "customer_name", "segment", "region_id", "signup_date"])
df_customer.write.saveAsTable("dim_customer")
print(f"✓ dim_customer: {df_customer.count()} rows")

# --- dim_date (2024 calendar) ---
spark.sql("DROP TABLE IF EXISTS dim_date")
date_rows = []
for d in range(366):  # 2024 is a leap year
    dt = date(2024, 1, 1) + timedelta(days=d)
    date_rows.append((
        dt,
        dt.year,
        dt.month,
        dt.day,
        (dt.month - 1) // 3 + 1,  # quarter
        dt.strftime('%B'),          # month_name
        dt.strftime('%A'),          # day_name
        dt.isocalendar()[1]         # week_of_year
    ))
df_date = spark.createDataFrame(date_rows, schema=["date_key", "year", "month", "day", "quarter", "month_name", "day_name", "week_of_year"])
df_date.write.saveAsTable("dim_date")


In [0]:
# --- fact_sales: 3000 rows ---
# The fact table contains foreign keys to dimensions + numeric measures only.
# This is the transactional grain: one row per order line item.

spark.sql("DROP TABLE IF EXISTS fact_sales")

random.seed(42)

fact_rows = []
for i in range(1, 3001):
    order_date = date(2024, 1, 1) + timedelta(days=random.randint(0, 365))
    product_id = random.randint(1, 15)
    customer_id = random.randint(1, 200)
    region_id = random.randint(1, 5)
    channel = random.choice(channels)
    unit_price = round(random.uniform(15, 500), 2)
    qty = random.randint(1, 6)
    discount = round(random.choice([0.0, 0.0, 0.0, 0.05, 0.10, 0.15, 0.20]), 2)
    net_rev = round(qty * unit_price * (1 - discount), 2)
    cost_val = round(net_rev * random.uniform(0.4, 0.7), 2)
    fact_rows.append((
        i, order_date, product_id, customer_id, region_id, channel,
        qty, unit_price, discount, net_rev, cost_val
    ))

df_fact = spark.createDataFrame(
    fact_rows,
    schema=["order_id", "order_date", "product_id", "customer_id", "region_id",
            "channel", "quantity", "unit_price", "discount_pct", "net_revenue", "cost"]
)
df_fact.write.saveAsTable("fact_sales")


In [0]:
# =============================================================
# ASSET TYPE 2: PRE-JOINED GOLD TABLE
# =============================================================
# A single denormalized table with all dimensions resolved.
# This is the simplest approach for dashboards — no JOINs needed at query time.
# Trade-off: data duplication and larger storage, but fastest queries.

spark.sql("DROP TABLE IF EXISTS gold_sales")

spark.sql(f"""
  CREATE TABLE gold_sales AS
  SELECT
    f.order_id,
    f.order_date,
    d.year           AS order_year,
    d.quarter        AS order_quarter,
    d.month_name     AS order_month_name,
    r.region_name    AS region,
    r.territory,
    p.product_name,
    p.category       AS product_category,
    p.subcategory    AS product_subcategory,
    c.customer_name,
    c.segment        AS customer_segment,
    f.channel,
    f.quantity,
    f.unit_price,
    f.discount_pct,
    f.net_revenue,
    f.cost,
    ROUND(f.net_revenue - f.cost, 2)  AS profit
  FROM fact_sales f
  JOIN dim_product  p ON f.product_id  = p.product_id
  JOIN dim_customer c ON f.customer_id = c.customer_id
  JOIN dim_region   r ON f.region_id   = r.region_id
  JOIN dim_date     d ON f.order_date  = d.date_key
""")

count = spark.sql("SELECT COUNT(*) FROM gold_sales").collect()[0][0]


In [0]:
%sql
-- =============================================================
-- ASSET TYPE 3: SQL VIEWS
-- =============================================================
-- Views encapsulate query logic without storing data.
-- They always reflect current source data (no refresh needed).
-- Use views to pre-shape data for specific dashboard use cases.

-- View 1: Monthly revenue summary (time-series ready)
CREATE OR REPLACE VIEW vw_monthly_revenue AS
SELECT
  DATE_TRUNC('MONTH', f.order_date)  AS order_month,
  r.region_name                      AS region,
  p.category                         AS product_category,
  COUNT(f.order_id)                  AS order_count,
  ROUND(SUM(f.net_revenue), 2)       AS total_revenue,
  ROUND(SUM(f.net_revenue - f.cost), 2) AS total_profit,
  ROUND(AVG(f.net_revenue), 2)       AS avg_order_value
FROM fact_sales f
JOIN dim_region  r ON f.region_id  = r.region_id
JOIN dim_product p ON f.product_id = p.product_id
GROUP BY ALL;

-- View 2: Region performance scorecard
CREATE OR REPLACE VIEW vw_region_performance AS
SELECT
  r.region_name,
  r.territory,
  COUNT(DISTINCT f.customer_id)       AS unique_customers,
  COUNT(f.order_id)                   AS total_orders,
  ROUND(SUM(f.net_revenue), 2)        AS total_revenue,
  ROUND(SUM(f.net_revenue - f.cost), 2) AS total_profit,
  ROUND(AVG(f.discount_pct) * 100, 1) AS avg_discount_pct,
  ROUND(SUM(f.net_revenue) / COUNT(DISTINCT f.customer_id), 2) AS revenue_per_customer
FROM fact_sales f
JOIN dim_region r ON f.region_id = r.region_id
GROUP BY r.region_name, r.territory;

-- View 3: Top products (ranked)
CREATE OR REPLACE VIEW vw_top_products AS
SELECT
  p.product_name,
  p.category,
  p.subcategory,
  COUNT(f.order_id)                      AS units_sold,
  ROUND(SUM(f.net_revenue), 2)           AS total_revenue,
  ROUND(SUM(f.net_revenue - f.cost), 2)  AS total_profit,
  ROUND((SUM(f.net_revenue) - SUM(f.cost)) / SUM(f.net_revenue) * 100, 1) AS profit_margin_pct,
  RANK() OVER (ORDER BY SUM(f.net_revenue) DESC) AS revenue_rank
FROM fact_sales f
JOIN dim_product p ON f.product_id = p.product_id
GROUP BY p.product_name, p.category, p.subcategory;

In [0]:
# =============================================================
# ASSET TYPE 4: METRIC VIEW
# =============================================================
# Metric views define reusable business KPIs in a semantic layer.
# Measures are pre-defined aggregations; dimensions are grouping attributes.
# Queries use MEASURE() syntax and GROUP BY ALL.
# This ensures consistent metric definitions across all dashboards.
#
# NOTE: Table references must be fully qualified (catalog.schema.table)
# so the metric view resolves correctly when queried from any context
# (e.g., dashboards, Genie spaces, other schemas).

spark.sql(f"""
CREATE OR REPLACE VIEW mv_sales_metrics
WITH METRICS
LANGUAGE YAML
AS $$
  version: 1.1
  source: {catalog}.{schema}.fact_sales
  joins:
    - name: prod
      source: {catalog}.{schema}.dim_product
      on: source.product_id = prod.product_id
    - name: cust
      source: {catalog}.{schema}.dim_customer
      on: source.customer_id = cust.customer_id
    - name: reg
      source: {catalog}.{schema}.dim_region
      on: source.region_id = reg.region_id
  dimensions:
    - name: Order Date
      expr: order_date
      comment: Date the order was placed
    - name: Order Month
      expr: DATE_TRUNC('MONTH', order_date)
      display_name: Order Month
      comment: Month of order for time-series analysis
      format:
        type: date
        date_format: locale_short_month
    - name: Order Quarter
      expr: QUARTER(order_date)
      display_name: Quarter
    - name: Region
      expr: reg.region_name
      display_name: Region
      comment: Sales region
    - name: Territory
      expr: reg.territory
      display_name: Territory
    - name: Product Name
      expr: prod.product_name
      display_name: Product
    - name: Product Category
      expr: prod.category
      display_name: Category
      synonyms:
        - category
        - product type
    - name: Customer Segment
      expr: cust.segment
      display_name: Segment
      synonyms:
        - segment
        - customer type
    - name: Channel
      expr: channel
      display_name: Sales Channel
  measures:
    - name: Total Revenue
      expr: SUM(net_revenue)
      display_name: Total Revenue
      comment: Sum of net revenue after discounts
      format:
        type: currency
        currency_code: USD
        decimal_places:
          type: exact
          places: 2
      synonyms:
        - revenue
        - sales
    - name: Total Profit
      expr: SUM(net_revenue - cost)
      display_name: Total Profit
      comment: Revenue minus cost
      format:
        type: currency
        currency_code: USD
        decimal_places:
          type: exact
          places: 2
    - name: Order Count
      expr: COUNT(order_id)
      display_name: Orders
      comment: Number of orders
    - name: Total Quantity
      expr: SUM(quantity)
      display_name: Units Sold
    - name: Avg Order Value
      expr: SUM(net_revenue) / COUNT(order_id)
      display_name: Avg Order Value
      comment: Average revenue per order
      format:
        type: currency
        currency_code: USD
        decimal_places:
          type: exact
          places: 2
    - name: Profit Margin
      expr: SUM(net_revenue - cost) / SUM(net_revenue)
      display_name: Profit Margin %
      comment: Profit as a percentage of revenue
      format:
        type: percentage
        decimal_places:
          type: exact
          places: 1
    - name: Avg Discount
      expr: AVG(discount_pct)
      display_name: Avg Discount
      format:
        type: percentage
        decimal_places:
          type: exact
          places: 1
    - name: Unique Customers
      expr: COUNT(DISTINCT customer_id)
      display_name: Unique Customers
      comment: Distinct customers who placed orders
$$""")

In [0]:
# --- Verification: confirm all assets exist ---
print("")
print("SETUP COMPLETE")
print("")
print(f"Catalog: {catalog}")
print(f"Schema:  {schema}")